# Model Benchmarking
# 模型基准测试

This tutorial demonstrates how to benchmark PipelineTS models across multiple datasets.
本教程展示如何在多个数据集上对 PipelineTS 模型进行基准测试。

Contents:
内容：

1. **Load multiple datasets / 加载多个数据集**
2. **Run ModelPipeline benchmarks / 运行 ModelPipeline 基准测试**
3. **Compare model performance across datasets / 跨数据集比较模型性能**
4. **Analyze results / 分析结果**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

## 1. Load Multiple Datasets
## 1. 加载多个数据集

PipelineTS provides several built-in datasets. We'll benchmark across multiple datasets.
PipelineTS 提供多个内置数据集。我们将在多个数据集上进行基准测试。

In [ ]:
from PipelineTS.dataset import (
    LoadElectricDataSets,
    LoadMessagesSentDataSets,
    LoadWebSales,
    LoadSupermarketIncoming,
    BuiltInSeriesData
)

# Define benchmark datasets / 定义基准测试数据集
benchmark_datasets = {
    'Electric': {
        'loader': LoadElectricDataSets,
        'time_col': 'date',
        'target_col': 'value',
    },
    'MessagesSent': {
        'loader': LoadMessagesSentDataSets,
        'time_col': 'date',
        'target_col': 'ta',
    },
    'WebSales': {
        'loader': LoadWebSales,
        'time_col': 'date',
        'target_col': 'type_a',
    },
    'Supermarket': {
        'loader': LoadSupermarketIncoming,
        'time_col': 'date',
        'target_col': 'goods_cnt',
    },
}

# Preview datasets / 预览数据集
for name, config in benchmark_datasets.items():
    df = config['loader']()
    print(f"{name}: shape={df.shape}, columns={df.columns.tolist()}")

## 2. Run Benchmarks with ModelPipeline
## 2. 使用 ModelPipeline 运行基准测试

We run ModelPipeline on each dataset and collect the leaderboard results.
我们在每个数据集上运行 ModelPipeline 并收集排行榜结果。

In [ ]:
from PipelineTS.pipeline import ModelPipeline

# Benchmark settings / 基准测试设置
LAGS = 12
MODELS = 'ml'  # Use ML models for speed / 使用 ML 模型以加快速度

all_results = {}

for ds_name, config in benchmark_datasets.items():
    print(f"\n{'='*50}")
    print(f"Benchmarking on {ds_name}... / 正在对 {ds_name} 进行基准测试...")
    print(f"{'='*50}")

    # Load and prepare data / 加载并准备数据
    df = config['loader']()
    time_col = config['time_col']
    target_col = config['target_col']
    df = df[[time_col, target_col]]
    df[time_col] = pd.to_datetime(df[time_col])

    # Create and train pipeline / 创建并训练管道
    pipeline = ModelPipeline(
        time_col=time_col,
        target_col=target_col,
        lags=LAGS,
        random_state=42,
        include_models=MODELS,
        metric=mean_absolute_error,
        metric_less_is_better=True,
        quantile=None,
        cv=3,
    )

    leaderboard = pipeline.fit(df)
    all_results[ds_name] = leaderboard
    print(f"\n{ds_name} leaderboard / {ds_name} 排行榜:")
    print(leaderboard)

## 3. Compare Results Across Datasets
## 3. 跨数据集比较结果

Create a summary table showing the best model for each dataset.
创建汇总表，显示每个数据集的最佳模型。

In [ ]:
# Create summary / 创建汇总
summary = []
for ds_name, leaderboard in all_results.items():
    best = leaderboard.iloc[0]
    summary.append({
        'Dataset / 数据集': ds_name,
        'Best Model / 最佳模型': best['model'],
        'MAE': f"{best['metric']:.4f}",
        'Train Time (s) / 训练时间': f"{best['train_cost(s)']:.1f}",
    })

summary_df = pd.DataFrame(summary)
print("Summary: Best model per dataset / 汇总：每个数据集的最佳模型")
print("=" * 70)
summary_df

In [ ]:
# Create a pivot table of MAE scores / 创建 MAE 分数透视表
pivot_data = []
for ds_name, leaderboard in all_results.items():
    for _, row in leaderboard.iterrows():
        pivot_data.append({
            'Dataset': ds_name,
            'Model': row['model'],
            'MAE': row['metric'],
        })

pivot_df = pd.DataFrame(pivot_data)
pivot_table = pivot_df.pivot_table(
    values='MAE', index='Model', columns='Dataset', aggfunc='first'
)

print("MAE pivot table (Model x Dataset) / MAE 透视表（模型 x 数据集）:")
pivot_table

## 4. Benchmark with Interval Prediction
## 4. 带区间预测的基准测试

We can also benchmark interval prediction accuracy (coverage rate).
我们也可以对区间预测精度（覆盖率）进行基准测试。

In [ ]:
# Benchmark with quantile prediction on Electric dataset
# 在电力数据集上进行分位数预测基准测试
df = LoadElectricDataSets()
df['date'] = pd.to_datetime(df['date'])

pipeline_q = ModelPipeline(
    time_col='date',
    target_col='value',
    lags=12,
    random_state=42,
    include_models='ml',
    metric=mean_absolute_error,
    metric_less_is_better=True,
    quantile=0.9,  # 90% prediction interval / 90% 预测区间
    cv=3,
)

leaderboard_q = pipeline_q.fit(df)
print("Leaderboard with interval prediction / 带区间预测的排行榜:")
leaderboard_q

## 5. Using BuiltInSeriesData for More Datasets
## 5. 使用 BuiltInSeriesData 获取更多数据集

`BuiltInSeriesData` provides access to additional datasets including ETT, M3, and AirPassengers.
`BuiltInSeriesData` 提供对更多数据集的访问，包括 ETT、M3 和 AirPassengers。

In [ ]:
from PipelineTS.dataset import BuiltInSeriesData

# List all available built-in datasets / 列出所有可用的内置数据集
series_data = BuiltInSeriesData()

# Access specific dataset / 访问特定数据集
etth1 = series_data['ETTh1']
print(f"ETTh1 shape: {etth1.shape}")
print(f"ETTh1 columns: {etth1.columns.tolist()}")
etth1.head()

## Summary / 总结

Key takeaways from benchmarking:
基准测试的关键结论：

- **No single model wins all**: Different models perform best on different datasets.
- **没有模型在所有数据集上都最优**：不同模型在不同数据集上表现最佳。

- **Use ModelPipeline**: Let the pipeline automatically find the best model for your data.
- **使用 ModelPipeline**：让管道自动为你的数据找到最佳模型。

- **Consider trade-offs**: Balance accuracy vs. training time based on your use case.
- **考虑权衡**：根据使用场景平衡精度和训练时间。

- **Interval prediction**: Use `quantile` parameter to get prediction intervals with coverage guarantees.
- **区间预测**：使用 `quantile` 参数获取具有覆盖率保证的预测区间。